# 19 · An ensemble on the p scale, and whether its verdicts are stable (Phase 5)

**Goal.** Build the final system from the **original recipe** (the owner's decision after the 13-fold evidence): three seeds, combined on the val-human tail-probability ("p") scale of notebook 18, with the threshold rule that the declared Phase 4 choice rule picked for the original recipe (rule (d), notebook 18b). Then check (1) that the Phase 4 threshold still means what it should for an ensemble, and (2) that the ensemble's verdicts are more stable than a single model's. The notebook ends with the **exact table of the single final held-out check**, which is defined here and **not computed** (the project owner sees its definition first).

**Everything in sections 1-7 was written and committed to git before any Phase 5 analysis was run.** The GPU jobs that produce the data (two more seeds of the original recipe on the full training set; leave-one-prompt-out fold 11 with three seeds) are prerequisites, not analyses, and were started first. The held-out prompts and held-out generators are not scored anywhere in this notebook before section 7 is signed off.

## 1 · The three models and where their scores come from

| seed | model | val and test scores |
|---|---|---|
| 42 | `models/distilroberta_finetuned` (notebook 08, the existing model; first 512 tokens) | `results/15_scores.csv`, column `roberta`, **only the val and test rows are used** (that file also holds held-out rows from Phase 1; they are not read) |
| 43 | `models/distilroberta_orig_s43` (same recipe, trained for this phase on all 28,006 training essays) | `results/17_final/orig_s43_scores.csv` (val and test only) |
| 44 | `models/distilroberta_orig_s44` (same) | `results/17_final/orig_s44_scores.csv` (val and test only) |

The recipe (notebook 08): DistilRoBERTa, 2 epochs, lr 2e-5, 6% warm-up, 512 tokens, fp16, batch 16, best epoch by val AUC. No augmentation, no weights.

## 2 · The p scale per seed, the ensemble, and the combination rule (chosen on val by a rule declared here)

- **Per seed:** $p_s(x) = (1 + \#\{\text{val humans with score} \ge x\}) / (1 + n)$ with the 2,291 val humans (all 13 ordinary prompts), exactly as in notebook 18. Small = more AI-like.
- **Ensemble score:** $e(x) = \text{combine}(p_{42}(x), p_{43}(x), p_{44}(x))$ with *combine* either the **mean** or the **median** of the three p-values.
- **Ensemble p:** $e$ is not itself a p-value, so it is re-calibrated the same way: $p_{\text{ens}}(x) = (1 + \#\{\text{val humans } h : e(h) \le e(x)\}) / (1 + n)$. All thresholds $\tau$ below are on this scale, so the three seeds, the ensemble and any single model are on one scale.
- **Choice of the combination (val only, rule declared before computing):** the combination with the higher share of **val AI essays with $p_{\text{ens}} \le 0.01$** (val AUC and the share caught at the 1% level are the only val statistics used; the val AI essays are not used for calibration, so this is not circular). If the two differ by less than **0.2 percentage points** (about two of the 1,210 val AI essays), the **mean** is chosen (declared tie-break: the simpler one). `test` and the held-out sets play no part in this choice.

## 3 · Testing the Phase 4 assumption: does the threshold carry over from single models to the ensemble?

Phase 4's thresholds were derived from **single-seed** leave-one-prompt-out models. The assumption that carries them to the ensemble (notebook 18, section 7): on a new topic, human essays land in the low-$p$ tail zone **at the same rates for the ensemble as for a single model**. Testable on two folds with three seeds of the original recipe:

- **Fold 7 (Facial action coding system)**: seeds 42, 43, 44 from notebook 16b (`results/16b_lopo`).
- **Fold 11 (Seeking multiple opinions)**: seeds 42, 43, 44 from this phase (`results/19_phase5/lopo`), chosen *by a rule declared now*: the prompt with the highest false-flag rate for the original recipe (5.80% in 16b) that does not already have three seeds, i.e. where an inflation effect is largest and the test is most informative. (Fold 11 is also the direct check on regression to the mean: its original-recipe rate with other seeds.)

For each fold, with each seed's p of the left-out humans computed against *that seed's own* other-prompt val humans (the same essays for every seed), and the ensemble's $p_{\text{ens}}$ computed from the seeds' p-values with the combination chosen in section 2 and calibrated on the same other-prompt val humans:

- the **false-flag rate at $p \le \tau$** for $\tau = 0.01$ and $0.005$ (nominal rates), for each single seed and for the ensemble, with Wilson intervals, and the **inflation factor** = observed rate / $\tau$;
- the **paired difference** (ensemble minus the average of the three singles) in false-flag rate, with a cluster bootstrap interval (1,000 resamples, seed 42) that redraws the left-out humans and the val humans.

**Reading, declared:** the assumption **holds** for a fold if the paired difference's interval is below or contains zero (the ensemble's inflation is no larger than a single model's). It **fails** if the interval lies above zero (the ensemble flags more new-topic humans than a single model at the same $\tau$): then $\tau$ would be too lax for the ensemble and would have to be re-derived. Both folds are reported separately; two topics are a small basis, and I say so.

## 4 · Stability

All at the **final operating point** $\tau(1\%) = 0.00235$ (rule (d), notebook 18b); the band edge $\tau(5\%) = 0.01032$ is used for the typo test below.

**(i) Agreement of verdicts across seeds.** Among essays flagged by **at least one** of the three single seeds, the share flagged by **all three** (the statistic of 16b, where it was 28 of 72), on the val and test humans, the val and test AI essays, and the left-out humans of folds 7 and 11. For the ensemble the analogue uses the three **leave-one-seed-out sub-ensembles** (two seeds each). *Caveat declared in advance:* each pair of sub-ensembles shares a seed, which inflates their agreement; this measure therefore flatters the ensemble and is **an upper bound, not a fair comparison**. Pairwise Cohen's kappa is reported as well.

**(ii) Flip rate under 5% typos for essays near the threshold.** Each essay is scored again after `inject_typos(text, 0.05)` (two independent typo copies per essay). *Near the threshold* = the essay's own (unperturbed) $p$ lies within a **factor of 3 of $\tau(1\%)$** on either side ($\tau/3 \le p \le 3\tau$; for a single model $p$ is that model's, for the ensemble it is $p_{\text{ens}}$). A **flip** = the verdict (flagged / not flagged at $\tau(1\%)$) differs between the original essay and its typo copy. Reported: the flip rate among near-threshold essays and among all essays; the single models (average over the three seeds) against the ensemble; a paired bootstrap interval for their difference. Data: the val and test essays (the three saved final models, scored here) and the left-out essays of fold 11 (the three fold models, scored in `scripts/lopo_fold_typos.py`, because fold models are not kept). Fold 7 has no typo scores.

## 5 · The final system's operating points and zones

The final system is the ensemble of the three original-recipe seeds with the combination chosen in section 2, and the thresholds are the ones the declared choice rule picked for the original recipe (notebook 18b, rule (d), from all 13 topics): **$\tau(0.5\%) = 0.00093$, $\tau(1\%) = 0.00235$, $\tau(5\%) = 0.01032$**. Zones: **flagged** $p_{\text{ens}} \le \tau_{\text{flag}}$; **cannot tell** (to be worded as proposed in 18b, "unusual but common for student essays") $\tau_{\text{flag}} < p_{\text{ens}} \le \tau(5\%)$; **not flagged** above. Two operating points: **A**: $\tau_{\text{flag}} = \tau(1\%)$; **B**: $\tau_{\text{flag}} = \tau(0.5\%)$; the band's upper edge is $\tau(5\%)$ in both.

## 6 · The single final held-out check: the exact table (defined here, NOT computed)

**Rule of use.** Run **once**, at the end of this phase, for the **final ensemble + thresholds + band together**, after the project owner has seen this definition. No change to the ensemble, the combination, the thresholds or the band after it has been run. The held-out prompts ("Does the electoral college work?", "Summer projects") and the held-out generators (`darragh_claude_v6`, `darragh_claude_v7`, `falcon_180b_v1`) are scored by the three final models and combined exactly as in section 2 (the val-human calibration is the same; nothing is re-fitted).

**Table H1, human essays of each held-out prompt, separately** (rows: Electoral college; Summer projects; both pooled). For operating point A ($\tau_{\text{flag}} = \tau(1\%)$) and B ($\tau(0.5\%)$): *n*; **% flagged**, **% in the cannot-tell band** and **% not flagged**, each with a 95% Wilson interval (the pooled row: cluster bootstrap, 1,000 resamples, seed 42). Two blocks: all human essays, and on-topic human essays only.

**Table H2, AI essays of the held-out prompts** (same rows, same two operating points): **% flagged (AI caught)**, **% in the band**, **% missed**; all essays and on-topic only.

**Table H3, AI essays of the held-out generators** (rows: `darragh_claude_v6`, `darragh_claude_v7`, `falcon_180b_v1`, pooled): **% flagged**, **% in the band**, **% missed**, both operating points; all essays and on-topic only.

**Table H4, context (not part of the claims):** the same human false-flag and AI-caught quantities for the **single seed-42 model at the old logit thresholds of notebook 09** (4.67 for "1%", 7.80 for "0.5%"), for continuity with the numbers already in the README and model card.

**Table H5, the check against what Phase 4 predicted.** Registered now, from `results/18b_*` (original recipe, rule (d), leave-one-topic-out over 13 topics, single-seed models):

| quantity (operating point A, $\tau(1\%)$) | prediction |
|---|---|
| mean human false-flag rate over new topics | 0.58% (95% interval 0.44-0.88) |
| human false-flag rate of any one new topic | within the range of the 13 left-out topics: **0.00% to 1.74%** (worst topic); 23% of topics above 1% |
| AI essays caught, new topics | 98.78% (97.74-99.21) |
| share of human / AI essays in the cannot-tell band | 2.49% / 0.69% |

H5 reports, for each held-out prompt, the observed human false-flag rate against the 0.00-1.74% range and the AI caught against 97.7-99.2%, and says plainly where an observation falls outside what was predicted. Summer projects was the hardest unseen prompt for the single original model (7.7% at the old 1% threshold), so a value near or above the top of the range there would be a finding, not a surprise.

**Not part of the check and not computed:** anything that would let the held-out results feed back into a choice.

## 7 · What is declared but not part of this notebook's claims

- The single-seed folds are the basis of the Phase 4 thresholds; the ensemble check in section 3 rests on **two topics** and three seeds each.
- The leave-one-seed-out sub-ensembles in 4(i) share seeds; the typo test in 4(ii) is the cleaner stability evidence.
- Seeds 43 and 44 of the full-training models are trained here; seed 42 is the existing model. There is no ensemble of more than three.

In [ ]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from src.data import load_splits
from src.evaluate import wilson_interval, ClusterSampler
from src.conformal import tail_p

SEED, N_BOOT = 42, 1000
TAU = {'0.5%': 0.00093, '1%': 0.00235, '5%': 0.01032}      # notebook 18b, rule (d), from all 13 topics (declared in section 5)
pd.set_option('display.width', 230); pd.set_option('display.max_colwidth', 100)
data = load_splits()

## 8 · The three models' scores and the combination (section 2 applied)

*(Everything from here on runs the analyses declared above; the declaration cells are unchanged.)*

In [ ]:
from sklearn.metrics import roc_auc_score
from src.ensemble import PEnsemble, inflation_fold, agreement, flips, wilson_pct, combine
from src.conformal import tail_p

vt = data[data['split'].isin(['val', 'test'])]
s42 = pd.read_csv('../results/15_scores.csv', index_col='row_id')['roberta']                      # the existing model; ONLY val/test rows are used below
f43 = pd.read_csv('../results/17_final/orig_s43_scores.csv', index_col='row_id')
f44 = pd.read_csv('../results/17_final/orig_s44_scores.csv', index_col='row_id')
assert set(f43['split']) <= {'val', 'test'} and set(f44['split']) <= {'val', 'test'}
sc = pd.DataFrame({42: s42.loc[vt.index], 43: f43['logit'].loc[vt.index], 44: f44['logit'].loc[vt.index]})
assert not sc.isna().any().any()
rows = []
for s in sc.columns:
    v, t = vt['split'] == 'val', vt['split'] == 'test'
    rows.append({'seed': s, 'val AUC': roc_auc_score(vt.loc[v, 'label'], sc.loc[v, s]), 'test AUC': roc_auc_score(vt.loc[t, 'label'], sc.loc[t, s]),
                 'val 1% logit threshold': float(np.quantile(sc.loc[v & (vt['label'] == 0), s], 0.99))})
print('the three seeds of the original recipe (first 512 tokens); the logit thresholds differ, the p scale removes that:')
pd.DataFrame(rows).set_index('seed').round(5)

In [ ]:
# section 2: choose the combination on val only, by the rule declared in section 2
vh = vt[(vt['split'] == 'val') & (vt['label'] == 0)].index
va = vt[(vt['split'] == 'val') & (vt['label'] == 1)].index
val_ai = {}
for how in ('mean', 'median'):
    ens = PEnsemble(sc.loc[vh].to_numpy(), how)
    val_ai[how] = 100 * float((ens.p_ens(sc.loc[va].to_numpy()) <= 0.01).mean())
single = {s: 100 * float((tail_p(sc.loc[va, s].to_numpy(), sc.loc[vh, s].to_numpy()) <= 0.01).mean()) for s in sc.columns}
print('val AI essays with p <= 0.01 (of', len(va), '):', {k: round(v, 2) for k, v in val_ai.items()}, '| single seeds:', {k: round(v, 2) for k, v in single.items()})
HOW = 'mean' if abs(val_ai['mean'] - val_ai['median']) < 0.2 else max(val_ai, key=val_ai.get)
print(f"difference mean - median: {val_ai['mean'] - val_ai['median']:+.2f} points  ->  combination chosen by the declared rule: {HOW}")

## 9 · Does the Phase 4 threshold carry over to the ensemble? (section 3, folds 7 and 11)

*(The combination used is the one chosen above on val; nothing here was used to choose it.)*

In [ ]:
FOLD_FILES = {7: ('Facial action coding system', '../results/16b_lopo/scores_fold07_seed{s}.csv'),
              11: ('Seeking multiple opinions', '../results/19_phase5/lopo/scores_fold11_seed{s}.csv')}
def fold_matrices(path_tpl, seeds=(42, 43, 44)):
    parts = {s: pd.read_csv(path_tpl.format(s=s), index_col='row_id').sort_index() for s in seeds}
    first = parts[seeds[0]]
    for s in seeds[1:]:
        assert first.index.equals(parts[s].index) and (first['role'].to_numpy() == parts[s]['role'].to_numpy()).all()
    vh_idx = first[(first['role'] == 'other_val') & (first['label'] == 0)].index
    lh_idx = first[(first['role'] == 'left_out') & (first['label'] == 0)].index
    la_idx = first[(first['role'] == 'left_out') & (first['label'] == 1)].index
    M = lambda idx: np.stack([parts[s].loc[idx, 'logit'].to_numpy() for s in seeds], axis=1)
    return dict(parts=parts, vh=vh_idx, lh=lh_idx, la=la_idx, val_h=M(vh_idx), left_h=M(lh_idx), left_a=M(la_idx))
inf_rows, FOLD = [], {}
for k, (name, tpl) in FOLD_FILES.items():
    FOLD[k] = fold_matrices(tpl)
    d = FOLD[k]
    r = inflation_fold(d['val_h'], d['left_h'], taus=(0.01, 0.005), how=HOW, val_clusters=data.loc[d['vh'], 'dup_cluster'].to_numpy(),
                       left_clusters=data.loc[d['lh'], 'dup_cluster'].to_numpy(), n_boot=N_BOOT, seed=SEED)
    inf_rows.append(r.assign(fold=k, topic=name))
inf = pd.concat(inf_rows, ignore_index=True)
inf.round(4).to_csv('../results/19_inflation_test.csv', index=False)
show = inf[['topic', 'tau', 'single seeds %', 'mean of singles %', 'ensemble %', 'inflation, singles (x nominal)', 'inflation, ensemble (x nominal)', 'ensemble minus singles (pts)', 'lo', 'hi', 'n']].copy()
show['assumption (section 3 reading)'] = np.where(show['hi'] < 0, 'holds (ensemble lower)', np.where(show['lo'] > 0, 'FAILS (ensemble higher)', 'holds (interval contains 0)'))
show.round(3)

## 10 · Stability (section 4)

### (i) Agreement of verdicts across seeds, at tau(1%) = 0.00235

*Deviation from the declaration, stated plainly:* the declaration lists "val and test humans", but the val humans are the **calibration set of the p scale itself**, so their p-values are circular; only the **test humans** are used among humans (the AI essays of val and test are fine: the AI essays are not used for calibration).

In [ ]:
TAU1 = TAU['1%']
def flags_for(X_single, ref_single, ref_how=HOW):
    """verdicts of each single seed (p_s <= tau) and of each leave-one-seed-out sub-ensemble (p_ens <= tau); X, ref: (n, 3) score matrices."""
    S = X_single.shape[1]
    ens_all = PEnsemble(ref_single, ref_how)
    single = ens_all.p_seeds(X_single) <= TAU1
    subs = []
    for drop in range(S):
        keep = [i for i in range(S) if i != drop]
        subs.append(PEnsemble(ref_single[:, keep], ref_how).p_ens(X_single[:, keep]) <= TAU1)
    return single, np.stack(subs, axis=1), ens_all.p_ens(X_single) <= TAU1
sets = {}
t_h = vt[(vt['split'] == 'test') & (vt['label'] == 0)].index; ai_idx = vt[(vt['label'] == 1)].index
ref_vt = sc.loc[vh].to_numpy()
sets['test humans'] = sc.loc[t_h].to_numpy(); sets['val + test AI essays'] = sc.loc[ai_idx].to_numpy()
refs = {'test humans': ref_vt, 'val + test AI essays': ref_vt}
for k, d in FOLD.items():
    sets[f'fold {k} left-out humans'] = d['left_h']; refs[f'fold {k} left-out humans'] = d['val_h']
rows = []
for name, X in sets.items():
    single, subs, full = flags_for(X, refs[name])
    a, b = agreement(single), agreement(subs)
    rows.append({'essays': name, 'n': len(X), 'single seeds: flagged by >=1': a['flagged by at least one'], 'by all 3': a['flagged by all'], 'share by all %': round(a['share flagged by all %'], 1), 'kappa': round(a['mean pairwise kappa'], 3),
                 'sub-ensembles: flagged by >=1': b['flagged by at least one'], 'by all 3': b['flagged by all'], 'share by all % (upper bound)': round(b['share flagged by all %'], 1), 'kappa (sub)': round(b['mean pairwise kappa'], 3)})
agree_tab = pd.DataFrame(rows).set_index('essays')
agree_tab.to_csv('../results/19_agreement.csv')
agree_tab

### (ii) Flip rate under 5% typos for essays near the threshold

The test essays and the val AI essays are scored again by the three final models after `inject_typos(text, 0.05)` (two independent copies each); for fold 11 the left-out essays were scored after typos inside `scripts/lopo_fold_typos.py` (the fold models are not kept). Each perturbed essay's $p$ is computed against the same **unperturbed** reference humans.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from src.paraphraser import inject_typos

TYPO_CACHE = '../results/19_phase5/typo_scores_test_valAI.csv'
eval_idx = vt[(vt['split'] == 'test') | ((vt['split'] == 'val') & (vt['label'] == 1))].index       # test essays + val AI essays (val humans are the reference)
if os.path.exists(TYPO_CACHE):
    typo = pd.read_csv(TYPO_CACHE, index_col='row_id')
else:
    typo = pd.DataFrame(index=eval_idx)
    texts = {c: [inject_typos(data.loc[i, 'text'], 0.05, seed=c * 1_000_003 + int(i)) for i in eval_idx] for c in (1, 2)}
    @torch.no_grad()
    def score(model_dir, text_list, batch_size=64):
        tok = AutoTokenizer.from_pretrained(model_dir); mdl = AutoModelForSequenceClassification.from_pretrained(model_dir).cuda().eval()
        ids = tok(text_list, truncation=True, max_length=512)['input_ids']
        out = np.empty(len(ids), dtype=np.float32); order = np.argsort([len(x) for x in ids], kind='stable')
        for s in range(0, len(order), batch_size):
            ch = order[s:s + batch_size]; w = max(len(ids[i]) for i in ch)
            x = torch.full((len(ch), w), tok.pad_token_id, dtype=torch.long)
            for r, i in enumerate(ch): x[r, :len(ids[i])] = torch.tensor(ids[i])
            with torch.autocast('cuda', dtype=torch.float16):
                out[ch] = mdl(input_ids=x.cuda(), attention_mask=(x != tok.pad_token_id).long().cuda()).logits.squeeze(1).float().cpu().numpy()
        del mdl; torch.cuda.empty_cache()
        return out
    for seed_, mdir in ((42, '../models/distilroberta_finetuned'), (43, '../models/distilroberta_orig_s43'), (44, '../models/distilroberta_orig_s44')):
        for c in (1, 2):
            typo[f's{seed_}_typo{c}'] = score(mdir, texts[c])
    typo.to_csv(TYPO_CACHE)
print(typo.shape)

In [ ]:
def flip_table(name, ref, X0, Xt, how=HOW):
    """X0: (n, 3) clean logits; Xt: list of two (n, 3) typo-copy logits. Flip rates, singles (each seed's own near set) and the ensemble."""
    ens = PEnsemble(ref, how)
    p0s, p0e = ens.p_seeds(X0), ens.p_ens(X0)
    ptS = [ens.p_seeds(x) for x in Xt]; ptE = [ens.p_ens(x) for x in Xt]
    stats_ = {}
    for s in range(3):
        stats_[f'seed {s}'] = flips(p0s[:, s], [p[:, s] for p in ptS], TAU1)
    stats_['ensemble'] = flips(p0e, ptE, TAU1)
    return stats_
def summarise_flips(name, stats_, n_boot=N_BOOT):
    n = len(next(iter(stats_.values()))[0])
    rng = np.random.default_rng(SEED)
    def rate(rows, key):
        fl, near = stats_[key]
        nn = near[rows]
        return (fl[rows][nn].mean() if nn.any() else np.nan), (fl[rows].mean())
    out = {}
    for key in stats_:
        r_near, r_all = rate(np.arange(n), key)
        out[key] = {'near-threshold essays': int(stats_[key][1].sum()), 'flip rate near %': 100 * r_near, 'flip rate all essays %': 100 * r_all}
    d_near, d_all = [], []
    for _ in range(n_boot):
        rows = rng.integers(0, n, n)
        rs = [rate(rows, f'seed {s}') for s in range(3)]; re = rate(rows, 'ensemble')
        d_near.append(100 * (re[0] - np.nanmean([x[0] for x in rs]))); d_all.append(100 * (re[1] - np.mean([x[1] for x in rs])))
    tab = pd.DataFrame(out).T
    tab.loc['singles (mean of 3)'] = [tab.loc[[f'seed {s}' for s in range(3)], 'near-threshold essays'].mean(), tab.loc[[f'seed {s}' for s in range(3)], 'flip rate near %'].mean(), tab.loc[[f'seed {s}' for s in range(3)], 'flip rate all essays %'].mean()]
    lo_n, hi_n = np.nanpercentile(d_near, [2.5, 97.5]); lo_a, hi_a = np.percentile(d_all, [2.5, 97.5])
    print(f"{name}: ensemble minus singles, flip rate near the threshold {np.nanmean(d_near):+.2f} pts [{lo_n:+.2f}, {hi_n:+.2f}]; all essays {np.mean(d_all):+.3f} pts [{lo_a:+.3f}, {hi_a:+.3f}]")
    return tab.round(3)
# test essays + val AI essays (references: the val humans, unperturbed)
X0 = sc.loc[typo.index].to_numpy()
Xt = [typo[[f's{s}_typo{c}' for s in (42, 43, 44)]].to_numpy() for c in (1, 2)]
ft_vt = flip_table('test + val AI', ref_vt, X0, Xt)
flip_vt = summarise_flips('test essays + val AI essays', ft_vt)
flip_vt

In [ ]:
# fold 11: left-out humans and AI essays after typos (the three fold models of scripts/lopo_fold_typos.py)
d = FOLD[11]
lo_idx = d['parts'][42][d['parts'][42]['role'] == 'left_out'].index
X0 = np.stack([d['parts'][s].loc[lo_idx, 'logit'].to_numpy() for s in (42, 43, 44)], axis=1)
Xt = [np.stack([d['parts'][s].loc[lo_idx, f'logit_typo{c}'].to_numpy() for s in (42, 43, 44)], axis=1) for c in (1, 2)]
ft11 = flip_table('fold 11', d['val_h'], X0, Xt)
flip11 = summarise_flips('fold 11 left-out essays (humans and AI)', ft11)
pd.concat({'test + val AI': flip_vt, 'fold 11 left-out': flip11}).to_csv('../results/19_flip_rates.csv')
flip11